# 08 — Generalization to a second cell: LG 18650HG2 (Google Colab)
**Question:** does the proposed IR-compensation channel, **with unchanged hyperparameters**, also work at an unseen −20 °C on a different cell (3 Ah NMC)?

**Protocol (same logic as for the Panasonic cell):** all models are **trained from scratch** on this cell (its OCV, capacity and resistance differ, so testing the Panasonic models would not be meaningful).
Training: Mixed 1–7 · validation (model selection / early stopping): Mixed 8 · training/validation temperatures 40/25/10/0/−10 °C ·
test: UDDS, HWFET, LA92, US06 × 6 temperatures · **−20 °C is never used in any stage**. Hyperparameters are those selected on the Panasonic cell (`01_hyperparameter_tuning`).

Results are reported in Section III-H and Table 6 of the article; the output tables are in `results/generalization_LG_HG2.xlsx`.

**Data:** P. Kollmeyer, M. Naguib, M. Skells, *LG 18650HG2 Li-ion Battery Data*, Mendeley Data, V2, 2020, doi: 10.17632/b5mj79w5w9.2 (not included in this repository).

**Google Drive layout (once):**
```
MyDrive/LG_generalization/
├── code/            ← the repository folders data_utils/ and models/ (with their contents)
├── selected.json    ← results/configs/selected_hyperparameters.json from this repository
└── LG_data/         ← the unzipped LG HG2 dataset (sub-folders as they are)
```
**Running:** runtime type **GPU** (A100 or L4) → run the cells in order. Check the file table printed by cell 2; if it is correct, set `CONFIRMED = True` in cell 0.
If the session is interrupted, **run all cells again**: loaded data come from the cache and finished training runs are skipped.
A short set-up test with `SMOKE = True` (a few minutes) is recommended first (then delete the `LG_runs` folder and set `SMOKE = False`).

In [ ]:
# ── 0. Settings (edit this cell only) ───────────────────────────────────────────────────────
BASE      = '/content/drive/MyDrive/LG_generalization'   # working folder on Google Drive
CODE_DIR  = f'{BASE}/code'            # contains data_utils/ and models/ (copied from this repository)
SELECTED  = f'{BASE}/selected.json'   # copy of results/configs/selected_hyperparameters.json
DATA_DIR  = f'{BASE}/LG_data'         # LG HG2 dataset (unzipped; any sub-folder layout)
OUT_DIR   = f'{BASE}/LG_runs'         # outputs (weights, run records, Excel, figure)

CAP_AH      = 3.0                     # LG HG2 nominal capacity → SoC = (CAP_AH + Ah) / CAP_AH
TRAIN_TEMPS = [40, 25, 10, 0, -10]    # training + validation temperatures; −20 °C is NEVER used (unseen)
TRAIN_MIX   = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7']   # training: Mixed 1–7
VAL_MIX     = ['M8']                  # validation (model selection / early stopping): Mixed 8
TEST_CYC    = ['UDDS', 'HWFET', 'LA92', 'US06']            # test: 4 standard cycles × 6 temperatures
LIGHT       = False                   # True: train on Mixed 1–4 only (close to the Panasonic data volume, ~2× faster)

SEEDS         = [101, 202, 303]
RUN_RCONST    = True                  # mechanism check: Proposed with temperature-independent R (recommended)
RUN_BASELINES = True                  # also train LSTM / GRU / CNN-LSTM / Transformer (~2× run time)
SMOKE         = False                 # True: very small quick run (set-up test, a few minutes)
CONFIRMED     = False                 # set True after checking the file table of cell 2
if LIGHT: TRAIN_MIX = ['M1', 'M2', 'M3', 'M4']

# Display names used in the released tables (internal names are kept by the code; see README)
NAMES = {'Önerilen (IR-telafili hibrit)': 'Proposed (IR-compensated hybrid)',
         'Hibrit (GRU→Transformer)': 'Hybrid (GRU→Transformer)',
         'C: R sıcaklıktan bağımsız': 'Proposed, temperature-independent R'}

In [ ]:
# ── 1. Drive, code, packages ───────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os, sys, re, json, time, shutil, pickle, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
try:
    import mat73  # noqa: F401  (for v7.3 .mat files)
except ImportError:
    os.system('pip -q install mat73')
shutil.rmtree('/content/code', ignore_errors=True)
shutil.copytree(CODE_DIR, '/content/code')           # code to the local disk (importing from Drive is slow)
sys.path.insert(0, '/content/code')
import numpy as np, pandas as pd, torch, scipy.io as sio
import data_utils.data as D
from data_utils.data import Normalizer
import models.experiment as E
from models import engine
for p in [DATA_DIR, SELECTED]:
    assert Path(p).exists(), f'Not found: {p}'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE → Runtime › Change runtime type › GPU')

In [ ]:
# ── 2. File discovery: temperature from folder/file name, cycle from file name ──────────────────
TEMP_RE = re.compile(r'(?<![0-9])(n|-|m|neg|minus)?\s*(\d{1,2})\s*deg', re.I)
CYC_RE  = re.compile(r'(?<![A-Za-z])(UDDS|HWFET|HWFTa?|HWY|LA92|US06|Mix(?:ed)?[\s_-]*([1-8]))(?![0-9])', re.I)
SKIP_RE = re.compile(r'charg|chg|pause|hppc|cap|c20|c_20|dis_|ocv', re.I)
SKIP_PATH = re.compile(r'prepared|norm|train|test', re.I)   # skip the pre-processed (normalized) subsets; raw files are used

def parse(path):
    rel = Path(path).relative_to(DATA_DIR)
    t = None
    for part in [rel.stem] + list(reversed(rel.parts[:-1])):
        m = TEMP_RE.search(part)
        if m: t = (-1 if m.group(1) else 1) * int(m.group(2)); break
    if SKIP_RE.search(rel.stem) or SKIP_PATH.search(str(rel.parent)): return None
    c = CYC_RE.search(rel.stem)
    if c is None or t is None: return None
    tok = c.group(1).upper()
    cyc = f'M{c.group(2)}' if c.group(2) else ('HWFET' if tok.startswith('HW') else tok)
    return t, cyc

def load_mat(fp):
    """Project loader first (meas struct); otherwise the first struct that has a Voltage field."""
    try:
        return D.load_raw(fp)
    except Exception:
        try:
            raw = sio.loadmat(str(fp), squeeze_me=True, struct_as_record=False)
        except Exception:
            import mat73; raw = mat73.loadmat(str(fp))
        for k, v in raw.items():
            if k.startswith('__'): continue
            get = (lambda f: getattr(v, f)) if hasattr(v, 'Voltage') else (lambda f: v[f]) if isinstance(v, dict) and 'Voltage' in v else None
            if get: return {f: np.asarray(get(f)).astype(float).ravel() for f in ['Time', 'Voltage', 'Current', 'Ah', 'Battery_Temp_degC']}
        raise KeyError(f'{fp}: no Voltage field found; keys {list(raw)}')

found = []
for fp in sorted(Path(DATA_DIR).rglob('*.mat')):
    r = parse(fp)
    if r: found.append({'T (°C)': r[0], 'cycle': r[1], 'file': str(fp.relative_to(DATA_DIR)), 'MB': fp.stat().st_size / 1e6})
F = pd.DataFrame(found)
assert len(F), f'No drive-cycle files found under {DATA_DIR}. Check the folder path.'
F['role'] = np.where(F['cycle'].isin(TEST_CYC), 'test',
            np.where(F['cycle'].isin(TRAIN_MIX) & F['T (°C)'].isin(TRAIN_TEMPS), 'train',
            np.where(F['cycle'].isin(VAL_MIX) & F['T (°C)'].isin(TRAIN_TEMPS), 'validation', '— not used')))
# several files for the same (temperature, cycle) → the largest one is used (warning)
F = F.sort_values(['T (°C)', 'cycle', 'MB'], ascending=[False, True, False])
dup = F[F.duplicated(['T (°C)', 'cycle'], keep=False)]
F = F.drop_duplicates(['T (°C)', 'cycle'])
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 200)
print(F.pivot_table(index='T (°C)', columns='cycle', values='role', aggfunc='first').sort_index(ascending=False).fillna('·').to_string())
print(f'\nTotal: train {(F.role == "train").sum()} · validation {(F.role == "validation").sum()} · test {(F.role == "test").sum()} files')
if len(dup): print('\n⚠️ Several files for the same temperature/cycle; the largest was used:\n', dup.to_string(index=False))
miss = [(t, c) for t in TRAIN_TEMPS + [-20] for c in TEST_CYC if not ((F['T (°C)'] == t) & (F['cycle'] == c)).any()]
if miss: print('\n⚠️ Missing test files:', miss)
print('\n→ If the table is correct, set CONFIRMED = True in cell 0 and run cell 0 and the following cells.')
print('  If not (e.g. a temperature could not be read), export the file list: F.to_csv("files.csv")')

In [ ]:
# ── 3. Loading → 1 Hz table (step-synchronous resampling, same pipeline as Panasonic) + cache + quality filter ──
assert CONFIRMED, 'Check the table of cell 2 and set CONFIRMED = True in cell 0.'
D.NOMINAL_CAPACITY = CAP_AH                    # SoC label with the LG nominal capacity
lab = lambda t: f'{t}degC'
use = F[F.role != '— not used']
cache = Path(OUT_DIR) / 'cache' / f'frames_by_file_cap{CAP_AH}.pkl'; cache.parent.mkdir(parents=True, exist_ok=True)
byfile = pickle.load(open(cache, 'rb')) if cache.exists() else {}      # key = file path (re-read if the file changes)
t0 = time.time()
for r in use.itertuples():
    if r.file in byfile: continue
    byfile[r.file] = D.to_frame(load_mat(Path(DATA_DIR) / r.file))
    print(f'✅ {lab(r[1]):>8s} {r[2]:6s} {len(byfile[r.file]):7,d} s · {time.time() - t0:.0f} s')
    pickle.dump(byfile, open(cache, 'wb'))
frames = {(lab(r[1]), r[2]): byfile[r.file] for r in use.itertuples()}

# Quality filter: records that are not drive cycles, are incomplete, or were run at the wrong temperature are removed
MIN_DIS_AH, MAX_DT = 0.5, 8.0                  # at least 0.5 Ah discharged; mean cell T within 8 °C of the chamber temperature
qc = []
for (t, c), d in frames.items():
    dis = -(d.ah.min() - d.ah.iloc[0]); Tm = float(d.temp.mean()); Tn = int(t[:-4])
    why = 'discharge < 0.5 Ah' if dis < MIN_DIS_AH else (f'cell T {Tm:.1f} °C ≠ chamber {Tn} °C' if abs(Tm - Tn) > MAX_DT else '')
    if why: qc.append({'temp': t, 'record': c, 'discharge Ah': round(dis, 2), 'T mean': round(Tm, 1), 'reason': why})
for q in qc: frames.pop((q['temp'], q['record']))
if qc: print('\n⚠️ The quality filter removed these records:\n' + pd.DataFrame(qc).to_string(index=False))

role_dfs = {k: d for k, d in frames.items() if k[1] in TRAIN_MIX + VAL_MIX and int(k[0][:-4]) in TRAIN_TEMPS}
te_dfs = {k: d for k, d in frames.items() if k[1] in TEST_CYC}
rows = []
for (t, c), d in sorted(frames.items()):
    rows.append({'split': 'test' if c in TEST_CYC else ('validation' if c in VAL_MIX else 'train'), 'temp': t, 'record': c, **D.label_check(d)})
file_table = pd.DataFrame(rows)
print('\nRecords used (train / validation / test):')
print(file_table.pivot_table(index='temp', columns='split', values='record', aggfunc=lambda x: ' '.join(sorted(x))).to_string())
miss = [(t, c) for t in TRAIN_TEMPS + [-20] for c in TEST_CYC if (lab(t), c) not in te_dfs]
if miss: print('\n⚠️ Missing from the test set:', miss, '→ see the list printed by cell 3b (the file may be named differently)')

In [ ]:
# ── 3b. (Diagnostics) If records are missing or were removed: ALL .mat files in that temperature folder ──
for t, c in sorted(set([(int(q['temp'][:-4]), q['record']) for q in qc] + miss)):
    hits = [fp for fp in Path(DATA_DIR).rglob('*.mat') if (parse(fp) or (None,))[0] == t or f'{abs(t)}deg' in str(fp).lower()]
    print(f'\n{t} °C — looking for: {c} · files in the folder:')
    for fp in sorted(set(hits)): print(f'   {str(fp.relative_to(DATA_DIR)):80s} {fp.stat().st_size / 1e6:6.2f} MB')

In [ ]:
# ── 4. Set up the experiment manager with the LG data (Panasonic code unchanged; roles M1–M8) ──────
E.TRAIN_ROLES = list(TRAIN_MIX); E.VAL_ROLES = list(VAL_MIX)
exp = E.Experiment(OUT_DIR, smoke=SMOKE)
exp.role_dfs, exp.te_dfs, exp.file_table, exp.pre_ms = role_dfs, te_dfs, file_table, 0.0
exp.norm = Normalizer().fit([d for (t, r), d in role_dfs.items() if r in TRAIN_MIX])
json.dump(exp.norm.stats, open(exp.p['configs'] / 'normalizer.json', 'w'), indent=1)
# selected_hyperparameters.json in this repository uses English model names; the code expects the internal names
INTERNAL = {'Proposed (IR-compensated hybrid)': 'Önerilen (IR-telafili hibrit)', 'Hybrid (GRU→Transformer)': 'Hibrit (GRU→Transformer)',
            'Proposed, temperature-independent R': 'C: R sıcaklıktan bağımsız', 'Proposed-S (h=64)': 'C: eşit boyut (h=64)',
            'RC-hybrid, no current input (former candidate)': 'Önerilen (RC-Hibrit)', 'RC-hybrid, no current input, h=64': 'Önerilen-küçük (h=64)',
            'Multi-scale RC extension (with current input)': 'RC + akım girişi', 'Hybrid without current input (noI)': 'Akımsız (noI)',
            'Ohmic only + current input (= Proposed, 03b training name)': 'RC: yalnız ohmik + akım girişi'}
sel_json = json.load(open(SELECTED, encoding='utf-8'))
json.dump({INTERNAL.get(k, k): v for k, v in sel_json.items()}, open(exp.p['configs'] / 'selected.json', 'w', encoding='utf-8'),
          ensure_ascii=False, indent=1)
sel = exp.load_selected()
PLAN = [(E.C_NAME, E.c_spec(sel)), ('Hibrit (GRU→Transformer)', sel['Hibrit (GRU→Transformer)'])]
if RUN_RCONST: PLAN.append(('C: R sıcaklıktan bağımsız', E.c_specs(sel)[0]['C: R sıcaklıktan bağımsız']))
if RUN_BASELINES: PLAN += [(m, sel[m]) for m in ['LSTM', 'GRU', 'CNN-LSTM', 'Transformer'] if m in sel]
print(D.window_counts(role_dfs, te_dfs, TRAIN_MIX, VAL_MIX, exp.cfg['W']).to_string(index=False))
print('Training temperature range (cell, normalizer):', round(exp.norm.stats['temp']['min'], 1), '…', round(exp.norm.stats['temp']['max'], 1), '°C')
print('Plan:', [NAMES.get(n, n) for n, _ in PLAN], '× seed', SEEDS, '| hyperparameters SAME as Panasonic (no re-tuning)')

In [ ]:
# ── 5. Training + test (resumes where it stopped; if interrupted, run cells 0→5 again) ──────────
# Seed-first order: all models finish for the first seed → an early comparison is possible.
t0 = time.time(); n = len(PLAN) * len(SEEDS); i = 0
for seed in (SEEDS[:1] if SMOKE else SEEDS):
    for name, spec in PLAN:
        i += 1
        exp.run(name, spec, seed, 'lg', series=(seed == SEEDS[0]),
                ablations=E.RC_ABLATIONS_TEST if spec['family'] == 'rc' else ())
        print(f'[{i}/{n}] {NAMES.get(name, name)} s{seed} · elapsed {(time.time() - t0) / 3600:.2f} h')
        torch.cuda.empty_cache()

In [ ]:
# ── 6. Results: seen (40…−10 °C) / unseen −20 °C, bias, paired tests ─────────────────────────
from scipy import stats
d = exp.collect('lg'); d['model'] = d['model'].map(lambda m: NAMES.get(m, m))
d['scope'] = np.where(d.temp == '-20degC', 'unseen −20 °C', 'seen')
C = NAMES[E.C_NAME]
sl = d.groupby(['model', 'scope', 'seed'])[['rmse', 'mae', 'maxe', 'bias']].mean() * 100
summ = sl.groupby(['model', 'scope']).agg(['mean', 'std']).round(3)
print('RMSE / MAE / MaxE / bias (% SoC, mean ± std over seeds):'); print(summ.to_string())
by_t = (d.groupby(['model', 'seed', 'temp']).rmse.mean() * 100).groupby(['model', 'temp']).mean().unstack('temp')
by_t = by_t[[c for c in ['40degC', '25degC', '10degC', '0degC', '-10degC', '-20degC'] if c in by_t]]
print('\nRMSE by temperature (%):'); print(by_t.round(2).to_string())
sig = []
for other in [NAMES.get(n, n) for n, _ in PLAN if n != E.C_NAME]:
    for k in d.scope.unique():
        m = d[d.scope == k]; c = m.groupby(['model', 'seed', 'temp', 'cycle']).rmse.mean() * 100
        if other not in c.index.get_level_values(0): continue
        a = c.loc[C]; b = c.loc[other].reindex(a.index); dd = (a - b).dropna()
        sig.append({'versus': other, 'scope': k, 'Proposed': a.mean(), 'model': b.mean(), 'Proposed better': f'{(dd < 0).sum()}/{len(dd)}',
                    'Wilcoxon p': stats.wilcoxon(dd).pvalue if (dd != 0).any() and len(dd) > 1 else np.nan})
sig = pd.DataFrame(sig); print('\nPaired comparison with Proposed (seed × temperature × cycle):'); print(sig.round(4).to_string(index=False))

In [ ]:
# ── 7. Learned R0(T) (Proposed) + Excel + figure ───────────────────────────────────────────
import matplotlib.pyplot as plt
st = exp.norm.stats['temp']; tn = lambda T: (T - st['min']) / (st['max'] - st['min'] + 1e-8)
Tcell = {t: np.median([float(np.median(v.temp)) for k, v in te_dfs.items() if k[0] == t]) for t in sorted(set(k[0] for k in te_dfs))}
rp = []
for rec in exp.records('lg'):
    p = rec.get('rc_params')
    if not p: continue
    for t, T in Tcell.items():
        rp.append({'model': NAMES.get(rec['name'], rec['name']), 'seed': rec['seed'], 'chamber': t, 'cell T median (°C)': T,
                   'R0 (mΩ)': float(np.exp(p['r0_a'] + p['r0_b'] * (tn(T) - 1)) * 1e3)})
rp = pd.DataFrame(rp)
if len(rp):
    r0 = rp.groupby(['model', 'chamber', 'cell T median (°C)'])['R0 (mΩ)'].agg(['mean', 'std']).reset_index().sort_values('cell T median (°C)', ascending=False)
    print('Learned R0 (at the measured cell temperature):'); print(r0.round(1).to_string(index=False))
# column names written by data_utils.data.label_check are internal (Turkish); translated for the released file
LABEL_COLS = {'süre (h)': 'duration (h)', 'örnek (1 Hz)': 'samples (1 Hz)', 'SoC baş': 'SoC start', 'SoC son': 'SoC end',
              'deşarj Ah': 'discharge Ah', 'şarj Ah (rejen.)': 'charge Ah (regen.)', 'V maks': 'V max',
              'T ort': 'T mean', 'T maks−min': 'T max−min'}
RAW_COLS = {'ofset': 'offset', 'sürüklenme': 'drift', 'artık_rmse': 'residual_rmse'}
with pd.ExcelWriter(Path(OUT_DIR) / 'generalization_LG_HG2.xlsx') as xw:
    summ.set_axis([f'{a} {b}' for a, b in summ.columns], axis=1).reset_index().to_excel(xw, sheet_name='Summary', index=False)
    by_t.reset_index().to_excel(xw, sheet_name='By temperature', index=False)
    sig.to_excel(xw, sheet_name='Significance', index=False)
    if len(rp): rp.to_excel(xw, sheet_name='R0', index=False)
    file_table.rename(columns=LABEL_COLS).to_excel(xw, sheet_name='Files', index=False)
    F.to_excel(xw, sheet_name='File mapping', index=False)
    d.rename(columns=RAW_COLS).to_excel(xw, sheet_name='Raw rows', index=False)
fig, ax = plt.subplots(figsize=(7.2, 2.8)); models = list(by_t.index); w = .8 / len(models); xs = np.arange(by_t.shape[1])
for i, m in enumerate(models):
    ax.bar(xs + (i - (len(models) - 1) / 2) * w, by_t.loc[m].values, w, label=m)
ax.set_xticks(xs); ax.set_xticklabels([c.replace('degC', ' °C') for c in by_t.columns]); ax.set_ylabel('RMSE (SoC %)')
ax.set_title('LG HG2 — −20 °C unseen'); ax.legend(fontsize=7, frameon=False); ax.grid(axis='y', alpha=.3)
fig.tight_layout(); fig.savefig(Path(OUT_DIR) / 'LG_rmse_by_temperature.png', dpi=300); plt.show()
print('✅', Path(OUT_DIR) / 'generalization_LG_HG2.xlsx')

### How to read the results
- **Main question:** at −20 °C, is Proposed clearly better than Hybrid (on the Panasonic cell: 1.82 vs 4.56 %)? Is the bias of Hybrid negative and that of Proposed ≈ 0?
- **Mechanism:** if the temperature-independent-R variant is worse than Proposed at −20 °C, the gain again comes from R(T) on this cell.
- **Seen temperatures:** Proposed ≈ Hybrid is expected (small differences).
- **R0 table:** the learned resistance should increase in the cold.
- Whatever the outcome, it is reported in the article (Section III-H, Table 6).